# Using SuperNeuroNAT to Code Logical Operations (NAND Gate)

When getting started using SuperNeuroMAT, it is useful to analyze basic problems and solve them using the neuron functions within SuperNeuroMAT that you will use later to complete more complex projects. Moving past the most basic logic gates of AND and OR, we can analyze the slightly more complex NAND and NOR gates to get used to using SuperNeuroMAT.

For those unfamiliar with these operations, they are defined as:
<br>NAND Gate: Given a set of two inputs, return true if AT LEAST ONE of the inputs is false
<br>NOR gate: Given a set of two inputs, returns true if NEITHER of the inputs is true.

In terms of application in the world of coding, here are the truth tables for each gate given x and y. 1 designates true, 0 designates false.

<br> For the NAND gate given two inputs:
<br>![diagram of NAND_gate](img/logic_gates_images/nand_gate_diagram.png "NAND_gate")
<br> For the NOR gate given two inputs:
<br>![diagram of NOR_gate](img/logic_gates_images/nor_gate_diagram.png "NOR_gate")

For this tutorial, we will focus on the NOR gate specifically.

In [6]:
### First, install and import superneuromat
#  use [pip install superneuromat inside of your terminal]
import superneuromat as snm
#  create a network for the NOR Gate
NOR_Gate=snm.SNN()
### Run this code to see the parameters of a neuron in SNM ###
help(NOR_Gate.create_neuron)

Help on method create_neuron in module superneuromat.neuromorphicmodel:

create_neuron(
    threshold: float = 0.0,
    leak: float = inf,
    reset_state: float = 0.0,
    refractory_period: int = 0,
    refractory_state: int = 0,
    initial_state: float | None = 0.0
) -> Neuron method of superneuromat.neuromorphicmodel.SNN instance
    Create a neuron in the SNN.

    Parameters
    ----------
    threshold : float, default=0.0
        Neuron threshold; the neuron spikes if its internal state is strictly
        greater than the neuron threshold
    leak : float, default=numpy.inf
        Neuron leak; the amount by which the internal state of the neuron is
        pushed towards its reset state
    reset_state : float, default=0.0
        Reset state of the neuron; the value assigned to the internal state
        of the neuron after spiking
    refractory_period : int, default=0
        Refractory period of the neuron; the number of time steps for which
        the neuron remains in

In our creation of this gate, we will utilize thresholds and weights to determine the function of the system. For this project, we will have two input neurons and one output neuron, along with a neuron that will constantly spike the output neuron regardless of our inputs. The input neurons will receive either a 0 or 1 from the user, with a 1 denoting a "spike". They will then communicate their information via "synapses" to the output neuron.
<br>For the NOR Gate, we will need 4 total neurons and three synapses. We will start creating the neurons as the next step. The input neurons need a threshold of 0, as they should spike no matter what input they receive. The NOR Gate should spike if neither input neuron spikes. Out of these 4 neurons, one of them will be designated as a constant spike so that, when neither input spikes, we can still have an output spike.

<br>![diagram of NOR_gate neurons](img/logic_gates_images/nor_gate_neurons.png "NOR_gate")

Here is a diagram of the neurons, their thresholds, and their connections. The numbers indicate the indices of specific neurons and the names designate neurons with special uses.

## STEP 1: Create Neurons

In [ ]:
inputs=[]
outputs=[]
###First, we create the two input neurons with threshold 0 and add them to the inputs list
for i in range(2):
    id=NOR_Gate.create_neuron(threshold=0)
    inputs.append(id)


###Next, we create the final output neuron with threshold 0 and adds it to the outputs list
id=NOR_Gate.create_neuron(threshold=0)
outputs.append(id)

###For the constant-spiking neuron, we want it to spike no matter what, so we can set its threshold to -1, so its reset state is greater than its threshold
# However, we need to add a refractory period of 1 (how long between when it can spike) so that we get outputs every other time step

constant=NOR_Gate.create_neuron(threshold=-1,refractory_period=1)

###Finally, we simulate the neural network and print it
print(NOR_Gate)



SNN with 4 neurons and 0 synapses @ 0x2015dcbdd30
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (4):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 []
     1           0           0         inf          0           0 []
     2           0           0         inf          0           0 []
     3           0          -1         inf          0           1 []

Synapse Info (0):
    idx    pre ->   post      weight    delay stdp_enabled


Input Spikes (0) for 0 time steps:
 Time:  Spike-value    Destination


Spike Train:
 t|id 
0 spikes since last reset


## STEP 2: Create Synapses

Next, we will take care of the synapses. For each synapse, we need to indicate the sending neuron, the receiving neuron, and the weight of the connection. As the input neurons and the constant neuron spike on the same time steps, we do not need any delays. The weight of the synapses between the inputs and the output are negative so that the reset state of the output neuron will be negative or zero if either one spikes.

In [ ]:
for i in range(2):
    #Creates synapses between the inputs and the output
    NOR_Gate.create_synapse(inputs[i],outputs[0],-1)

#Creates a synapse between the constant-spiking neuron and the output
NOR_Gate.create_synapse(constant,outputs[0],1)


print(NOR_Gate)


SNN with 4 neurons and 3 synapses @ 0x2015dcbdd30
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (4):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 []
     1           0           0         inf          0           0 []
     2           0           0         inf          0           0 []
     3           0          -1         inf          0           1 []

Synapse Info (3):
    idx    pre ->   post      weight    delay stdp_enabled
      0      0 ->      2          -1       1 -
      1      1 ->      2          -1       1 -
      2      3 ->      2           1       1 -

Input Spikes (0) for 0 time steps:
 Time:  Spike-value    Destination


Spike Train:
 t|id 
0 spikes since last reset


## STEP 3: Add Spikes

Finally, we need to add the spikes. These spikes will be 1's. As we are working on the NOR gate, if neither neuron spikes, the output neuron will spike. To add the spikes, we specify time step, neuron to be spiked, and the value of the spike.  Once the spikes for each combination of two inputs are added, we need to simulate the program to see its function. As each transfer from neuron to synapse takes one time step, we need to simulate 8 total time steps (starting at 0, gives us simulate(8)) to see every combination of inputs and outputs for the program.
<br>To see the results, look at the first three neurons in the spike train. The first two (indices 0,1) are the inputs and the third (index 2) is the output. This train should match the truth table for the NOR Gate from the beginning of this tutorial

In [9]:
NOR_Gate.add_spike(0,inputs[0],0)
NOR_Gate.add_spike(0,inputs[1],0)

NOR_Gate.add_spike(2,inputs[0],1)
NOR_Gate.add_spike(2,inputs[1],0)

NOR_Gate.add_spike(4,inputs[0],0)
NOR_Gate.add_spike(4,inputs[1],1)

NOR_Gate.add_spike(6,inputs[0],1)
NOR_Gate.add_spike(6,inputs[1],1)

NOR_Gate.simulate(8) 
print(NOR_Gate)


SNN with 4 neurons and 3 synapses @ 0x2015dcbdd30
STDP is globally enabled
apos: []
aneg: []
0 synapses have STDP enabled.

Neuron Info (4):
   idx       state      thresh        leak  ref_state  ref_period spikes
     0           0           0         inf          0           0 [--┴---┴-]
     1           0           0         inf          0           0 [----┴-┴-]
     2          -1           0         inf          0           0 [-┴------]
     3           0          -1         inf          0           1 [┴-┴-┴-┴-]

Synapse Info (3):
    idx    pre ->   post      weight    delay stdp_enabled
      0      0 ->      2          -1       1 -
      1      1 ->      2          -1       1 -
      2      3 ->      2           1       1 -

Input Spikes (0) for 0 time steps:
 Time:  Spike-value    Destination


Spike Train:
t|id0 1 2 3  
0: [│ │ │ ├─]
1: [│ │ ├─│ ]
2: [├─│ │ ├─]
3: [│ │ │ │ ]
4: [│ ├─│ ├─]
5: [│ │ │ │ ]
6: [├─├─│ ├─]
7: [│ │ │ │ ]
9 spikes since last reset


If you have not already done so, try to create the NAND Gate on your own. Then, try the NAND gate tutorial, also within this GitHub, to see the solution.

In [10]:
###Put your code below